# 生成式AI 第十週作業 — 用 Colab 建立 LLM Wiki 與教學 HTML 簡報

**主題**：AI Agents 的循環與圖（Loop & Graph）

本筆記本依課程三步驟流程執行：
1. **Ingest**：從 5 篇 arXiv 論文抓取標題與摘要，寫入 `wiki/` 知識庫
2. **整理重點 → 檢查內容**：先討論（loop/graph 角色表）、後生成，並用 Validator 檢查
3. **生成 HTML 簡報**：由 `wiki/` 內容自動產生 `index.html`（8 頁教學簡報）
4. **發布與驗證**：GitHub Pages 上線連結 ＋ 頁面預覽

作者：海大資工所 白東衢（生成式AI 課程）


In [ ]:
# %%cell【Ingest】步驟一：抓取 5 篇 arXiv 論文 → 寫入 wiki/ 知識庫
import urllib.request, re, os, json

SOURCES = {
    'react':       'https://arxiv.org/abs/2210.03629',  # ReAct: Reasoning + Acting
    'reflexion':   'https://arxiv.org/abs/2303.11366',  # Reflexion: verbal self-correction
    'toolformer':  'https://arxiv.org/abs/2302.04761',  # Toolformer: tool use
    'genagents':   'https://arxiv.org/abs/2304.03442',  # Generative Agents: memory+reflection
    'crag':        'https://arxiv.org/abs/2401.15884',  # CRAG: corrective RAG
}
# 各篇在 Agent loop/graph 中的角色（先討論的結論，寫入 Wiki）
ROLE = {
    'react':      '核心循環：思考 → 行動 → 觀察的迭代往返迴圈',
    'reflexion':  '反思節點：失敗後生成文字回饋重試（內層 loop）',
    'toolformer': '工具行動節點：技能擴充 Agent 可做的事',
    'genagents':  '記憶架構：觀察沉澱 → 反思 → 循環索取出題',
    'crag':       '檢查門檻：檢索品質分級與重新整理（校驗節點）',
}

os.makedirs('wiki', exist_ok=True)
records = []
for name, url in SOURCES.items():
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    html = urllib.request.urlopen(req, timeout=30).read().decode('utf-8', 'ignore')
    title = re.search(r'<meta name="citation_title" content="(.*?)"', html).group(1)
    authors = re.findall(r'<meta name="citation_author" content="(.*?)"', html)[:3]
    date = re.search(r'<meta name="citation_date" content="(.*?)"', html)
    absm = re.search(r'<meta name="citation_abstract" content="(.*?)"', html, re.S)
    abstract = re.sub(r'\s+', ' ', absm.group(1)).strip() if absm else ''
    with open(f'wiki/{name}.md', 'w', encoding='utf-8') as f:
        f.write(f'# {title}\n\n- 作者：{"; ".join(authors)}\n- 日期：{date.group(1) if date else ""}\n- 來源：{url}\n\n## 在 Agent loop/graph 的角色\n{ROLE[name]}\n\n## Abstract\n{abstract}\n')
    records.append({'name': name, 'title': title, 'url': url})
    print(f'[ingest] wiki/{name}.md  <-  {title[:60]}')

# 產生 index.md 目錄
with open('wiki/index.md', 'w', encoding='utf-8') as f:
    f.write('# LLM Wiki 目錄：AI Agents 的循環與圖\n\n')
    for r in records:
        f.write(f'- [{r["title"]}]({r["name"]}.md)（{r["url"]}）\n')
print('[ingest] wiki/index.md 已產生')
json.dump(records, open('wiki/records.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)


## 步驟二【先討論、後生成】整理重點 → 檢查內容

**討論結論**（已寫入各篇 Wiki 的「loop/graph 角色」段落）：

| 論文 | 在 Agent loop / graph 中的角色 |
|---|---|
| ReAct | 核心循環：思考 → 行動 → 觀察的迭代往返迴圈 |
| Reflexion | 反思節點：失敗後生成文字回饋重試（內層 loop） |
| Toolformer | 工具行動節點：技能擴充 Agent 可做的事 |
| Generative Agents | 記憶架構：觀察沉澱 → 反思 → 循環索取出題 |
| CRAG | 檢查門檻：檢索品質分級與重新整理（校驗節點） |

下一步用 Validator 檢查 5 篇 Wiki 是否齊全（標題＋摘要），通過才准生成簡報。


In [ ]:
# %%cell【檢查內容】Validator：確認 5 篇 Wiki 完整（執行與檢查紀錄）
ok = True
for name in SOURCES:
    t = open(f'wiki/{name}.md', encoding='utf-8').read()
    has_title = t.startswith('# ') and len(t.split('#')[1].strip()) > 5
    has_role = '## 在 Agent loop/graph 的角色' in t
    has_abs = '## Abstract' in t and len(t.split('## Abstract')[1]) > 100
    print(f'[check] wiki/{name}.md  標題={has_title} 角色={has_role} 摘要={has_abs}')
    ok = ok and has_title and has_role and has_abs
print()
print('=== 檢查結果：', 'PASS 5/5，可作為簡報知識庫' if ok else 'FAIL：請重跑 Ingest')
assert ok


In [ ]:
# %%cell【生成 HTML 簡報】步驟三：由 wiki/ 自動產生 index.html（8 頁）
import html as _html, textwrap

pages = []
pages.append(('cover', '<h1>AI Agents 的循環與圖</h1><p class="sub">從 5 篇論文看懂 Agent 的 loop 與 graph</p><p class="sub">生成式AI 第十週作業 ｜ 海大資工所 白東衢</p>'))
pages.append(('why', '<h2>為什麼 Agent 是「循環」不是「一句話」</h2><ul><li>單次問答：問一次、答一次，錯了不會修</li><li>Agent：<b>思考 → 行動 → 觀察</b>不斷往復，直到完成</li><li>多個能力（檢索、工具、反思）用<b>圖</b>串接，每個方塊是一個節點</li></ul>'))
for r in records:
    t = open(f'wiki/{r["name"]}.md', encoding='utf-8').read()
    title = t.splitlines()[0][2:]
    role = t.split('## 在 Agent loop/graph 的角色')[1].split('##')[0].strip()
    absm = t.split('## Abstract')[1].strip()
    pages.append((r['name'], f'<h2>{_html.escape(title)}</h2><p class="role"><b>角色：</b>{_html.escape(role)}</p><p class="abs">{_html.escape(textwrap.shorten(absm, 320))}</p><p class="src">來源：{_html.escape(r["url"])}</p>'))
pages.append(('graph', '<h2>把 5 篇拼成一張圖</h2><pre>  使用者提問<br>     │<br>     ▼<br> ┌► ReAct 思考 ──► 工具行動(Toolformer) ──► 觀察<br> │      │                                      │<br> │      ▼                                      ▼<br> │   CRAG 檢查檢索品質 ◄──────────────── 記憶沉澱(Generative Agents)<br> │      │                                      │<br> │      ▼                                      ▼<br> └── 失敗：Reflexion 反思重試 ◄─────── 成功：輸出答案</pre><p class="sub">內圈＝ReAct 迴圈；外圈＝反思與記憶的長程迴圈；CRAG＝门口的品質門檻</p>'))
pages.append(('end', '<h2>重點回顧</h2><ul><li>Agent 的心臟是<b>迴圈</b>（ReAct），不是更長的一次性提示</li><li><b>反思</b>讓迴圈越跑越好（Reflexion）</li><li><b>工具</b>擴大行動半徑（Toolformer）；<b>記憶</b>讓經驗可重複提取（Generative Agents）</li><li><b>檢查節點</b>把品質問題擋在生成之前（CRAG）</li></ul><p class="sub">知識庫：wiki/（本 Colab 產生）｜ 本簡報由 Colab 自動生成</p>'))

CSS = """<style>body{margin:0;background:#0d1420;font-family:'Noto Sans TC',sans-serif}
.sec{min-height:92vh;margin:4vh auto;max-width:900px;padding:6vh 6vw;background:#17233a;border:1px solid #3d5473;border-radius:14px;color:#e8eef8;box-shadow:0 10px 30px rgba(0,0,0,.4)}
h1{font-size:2.6em;color:#7fd1ff}h2{font-size:1.8em;color:#7fd1ff;border-bottom:2px solid #3d5473;padding-bottom:.3em}
.sub{color:#9db4d0}.role{background:#22334f;padding:.8em 1em;border-left:4px solid #7fd1ff;border-radius:6px}
.abs{color:#c7d6ea;line-height:1.7}.src{color:#6d84a3;font-size:.85em}pre{color:#a8e6a3;line-height:1.5;overflow-x:auto}
ul li{margin:.6em 0;line-height:1.6}nav{position:fixed;right:1em;top:1em}.dot{display:inline-block;width:10px;height:10px;margin:2px;background:#3d5473;border-radius:50%}</style>"""

parts = [f'<!DOCTYPE html><html lang="zh-Hant"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width, initial-scale=1.0"><title>AI Agents 的循環與圖 — LLM Wiki 教學簡報</title>{CSS}</head><body>',
         '<nav>' + ''.join('<span class="dot"></span>' for _ in pages) + '</nav>']
for key, body in pages:
    parts.append(f'<section class="sec" id="{key}">{body}</section>')
parts.append('</body></html>')
open('index.html', 'w', encoding='utf-8').write('\n'.join(parts))
print(f'[generate] index.html 已生成：{len(pages)} 頁（知識內容全部取自 wiki/）')


## 步驟四【發布與驗證】

- 本筆記本與 `wiki/`、`index.html` 同步於 GitHub 倉庫：
  **https://github.com/NextBai/nextbai-w10-llm-wiki**
- HTML 簡報上線連結（GitHub Pages）：
  **https://nextbai.github.io/nextbai-w10-llm-wiki/**
- 下方直接預覽簡報第一頁（iframe）


In [ ]:
# %%cell【驗證】顯示本機 index.html 前 30 行 ＋ 統計
txt = open('index.html', encoding='utf-8').read()
print('頁數:', txt.count('<section'))
print('字數:', len(txt))
print('\n'.join(txt.splitlines()[:6]), '\n...')


In [ ]:
# %%cell【預覽】在 Colab 內直接看 HTML 簡報
from google.colab import output as _out
_htmlsrc = open('index.html', encoding='utf-8').read()
_out.eval_js('new Promise(r=>{const f=document.createElement("iframe");f.style="width:100%;height:640px;border:1px solid #555;border-radius:10px";f.srcdoc=arguments[0];document.body.appendChild(f);r("iframe 已嵌入");})', _htmlsrc)
